# Data cleaning
 This script uploads and cleans the data for the Econ-ML project.

#### Libraries

In [ ]:
import pandas as pd


Connect to Google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import_grav_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Initial/Gravity_dta_V202211/'
import_sanc_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Initial/gsdb_v4/'
export_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/'

## Gravity
link to the source:
 https://www.cepii.fr/cepii/en/bdd_modele/bdd_modele_item.asp?id=8

In [ ]:
%%time
data_grav = pd.read_stata(import_grav_path + 'Gravity_V202211.dta')

CPU times: user 29.6 s, sys: 22.8 s, total: 52.4 s
Wall time: 1min 4s


In [ ]:
### Keeping only relevant variables

# redundant encodings of the same few concepts —
# five different distance measures, GDP in current vs. PPP vs.
# Penn-World-Table versions, multiple population sources are dropped.

keep_rename = {
    # --- Keys: identify each directed dyad-year (merge + fixed effects) ---
    "year":                 "year",           # year of observation
    "iso3_o":               "exp_iso3",       # exporter (origin) ISO3      <-- GSDB merge key
    "iso3_d":               "imp_iso3",       # importer (destination) ISO3 <-- GSDB merge key
    "country_id_o":         "exp_id",         # CEPII exporter id (tracks territorial changes over time)
    "country_id_d":         "imp_id",         # CEPII importer id

    # --- Outcome: bilateral trade flow ---
    "tradeflow_baci":       "trade",          # MAIN OUTCOME: BACI harmonised trade, 1995+, thousand USD
    "tradeflow_comtrade_o": "trade_comtrade", # robustness: UN Comtrade, exporter-reported (longer coverage)

    # --- Geography / distance (time-invariant; absorbed by pair FE, used in tree/DML) ---
    "dist":                 "dist",           # distance between capital cities (km)
    "distcap":              "dist_cap",       # capital-to-capital distance (alt. capital measure)
    "distw_harmonic":       "dist_w_harm",    # population-weighted distance, harmonic mean (preferred)
    "distw_arithmetic":     "dist_w_arith",   # population-weighted distance, arithmetic mean
    "distw_harmonic_jh":    "dist_w_harm_jh", # pop-weighted harmonic, Julian Hinz recomputation
    "distw_arithmetic_jh":  "dist_w_arith_jh",# pop-weighted arithmetic, Julian Hinz recomputation
    "contig":               "contig",         # =1 if the two countries share a land border

    # --- Cultural / historical ties (time-invariant) ---
    "comlang_off":          "comlang",        # =1 if common official/primary language
    "comcol":               "comcol",         # =1 if pair had a common colonizer after 1945
    "col45":                "colony",         # =1 if pair ever in a colonial relationship post-1945

    # --- Economic size (country-time; absorbed by country-time FE, used in tree/DML) ---
    "gdp_o":                "exp_gdp",        # exporter GDP (current USD)
    "gdp_d":                "imp_gdp",        # importer GDP (current USD)
    "gdpcap_o":             "exp_gdppc",      # exporter GDP per capita
    "gdpcap_d":             "imp_gdppc",      # importer GDP per capita
    "pop_o":                "exp_pop",        # exporter population
    "pop_d":                "imp_pop",        # importer population

    # --- Trade-policy variables (time-varying & dyadic: THESE survive the fixed
    #     effects and are what the sanction dummies compete against in the LASSO) ---
    "rta_coverage":         "rta",            # regional trade agreement in force (depth/coverage code)
    "fta_wto":              "fta",            # =1 if a WTO-notified FTA is in force for the pair
    "eu_o":                 "exp_eu",         # =1 if exporter is an EU member that year
    "eu_d":                 "imp_eu",         # =1 if importer is an EU member that year
    "wto_o":                "exp_wto",        # =1 if exporter is a GATT/WTO member that year
    "wto_d":                "imp_wto",        # =1 if importer is a GATT/WTO member that year
}

# Defensive subset (skips anything not present, e.g. if you did a slim read earlier)
present = [c for c in keep_rename if c in data_grav.columns]
missing = [c for c in keep_rename if c not in data_grav.columns]
if missing:
    print("Not found, skipped:", missing)

data_grav = data_grav[present].rename(columns={k: keep_rename[k] for k in present})

# Restrict to the BACI window: the main trade outcome only exists from 1995
data_grav = data_grav[data_grav["year"] >= 1995].reset_index(drop=True)

print(data_grav.shape)
data_grav.sample(3)

(1714608, 29)


,year,exp_iso3,imp_iso3,exp_id,imp_id,trade,trade_comtrade,dist,dist_cap,dist_w_harm,...,exp_gdppc,imp_gdppc,exp_pop,imp_pop,rta,fta,exp_eu,imp_eu,exp_wto,imp_wto
1156189,2017,OMN,USA,OMN,USA,753372.469,340232.596,11367.0,11692.0,12503.0,...,17.329,59.915001,4665.926,325122.131,Goods & Services,1.0,0.0,0.0,1.0,1.0
159004,1996,BHR,GUF,BHR,GUF,NaN,NaN,11055.0,11055.0,11055.0,...,10.526,NaN,579.697,150.933,NaN,0.0,0.0,0.0,1.0,0.0
595621,1996,GNB,MDA,GNB,MDA,NaN,NaN,5690.0,5690.0,5690.0,...,0.232,0.462000,1165.465,3667.748,NaN,0.0,0.0,0.0,1.0,0.0


## Sanctions
link to the source: https://www.globalsanctionsdatabase.com

In [ ]:
%%time
data_sanc = pd.read_stata(import_sanc_path + 'GSDB_V4_dyadic.dta')

CPU times: user 896 ms, sys: 215 ms, total: 1.11 s
Wall time: 1.66 s


In [ ]:
print(data_sanc.columns.tolist())

['case_id', 'sanctioning_state_iso3', 'sanctioning_state', 'sanctioned_state_iso3', 'sanctioned_state', 'year', 'arms', 'military', 'trade', 'descr_trade', 'financial', 'travel', 'other', 'target_mult', 'sender_mult', 'objective', 'success']


Fixing the old Belarus code

In [ ]:
# BYS is Belarus (old ISO code) -> match Gravity's BLR, both sides for safety
for col in ["sanctioning_state_iso3", "sanctioned_state_iso3"]:
    data_sanc[col] = data_sanc[col].replace({"BYS": "BLR"})

Checking whether  '' in the merge column is present a lot

In [ ]:
print((data_sanc["sanctioned_state_iso3"]=="").sum()/data_sanc.shape[0]) # 5%, a lot

0.05573193348631063


In [ ]:
blank = data_sanc[data_sanc["sanctioned_state_iso3"]==""]
print(blank["sanctioned_state"].value_counts().head())   # what country name sits behind the blank code?
print(blank["objective"].value_counts().head())

sanctioned_state
Terrorist Organizations (Al-Qaeda)                2509
Terrorist Organizations (Taliban)                 2509
Terrorist Organizations (ISIL and ANF)            1930
Terrorist Organizations (Taliban and Al-Qaeda)    1917
Name: count, dtype: int64
objective
terrorism    8865
Name: count, dtype: int64


So, they are not states- they are terrorist organizations

## Preliminary diagnostics

## Sanctions dataset

#### Types

In [ ]:
# GSDB 'year' came in as datetime; reduce it to an integer year to match Gravity
if pd.api.types.is_datetime64_any_dtype(data_sanc["year"]):
    data_sanc["year"] = data_sanc["year"].dt.year
data_sanc["year"] = data_sanc["year"].astype("int64")

# also make Gravity's year the same width, just to be safe
data_grav["year"] = data_grav["year"].astype("int64")

print(data_grav["year"].dtype, data_sanc["year"].dtype)   # both should say int64

int64 int64


#### Duplicates

In [ ]:
print("sanc dup keys:",
      data_sanc.duplicated(["sanctioning_state_iso3","sanctioned_state_iso3","year"]).sum())

sanc dup keys: 4632


Unfortunately, there are duplicates in the sanctions dataset: sender–target–year; dyad-year is not unique

In [ ]:
# peek at a few duplicated dyad-years to see how the rows differ
dups = data_sanc[data_sanc.duplicated(
    ["sanctioning_state_iso3","sanctioned_state_iso3","year"], keep=False)]
print(dups.shape)
dups.sort_values(["sanctioning_state_iso3","sanctioned_state_iso3","year"]).head(3)

(7141, 17)


,case_id,sanctioning_state_iso3,sanctioning_state,sanctioned_state_iso3,sanctioned_state,year,arms,military,trade,descr_trade,financial,travel,other,target_mult,sender_mult,objective,success
553,918,AFG,Afghanistan,,Terrorist Organizations (Al-Qaeda),2011,1,1,0,,1,1,0,1,1,terrorism,ongoing
585,690,AFG,Afghanistan,,Terrorist Organizations (Taliban and Al-Qaeda),2011,1,1,0,,1,1,0,1,1,terrorism,failed
586,919,AFG,Afghanistan,,Terrorist Organizations (Taliban),2011,1,1,0,,1,1,0,1,1,terrorism,ongoing


In [ ]:
dups = data_sanc[data_sanc.duplicated(
    ["sanctioning_state_iso3","sanctioned_state_iso3","year"], keep=False)]

# how many duplicate rows have a BLANK target (terrorist orgs) vs a real country code?
blank_target = (dups["sanctioned_state_iso3"] == "")
print("duplicate rows with blank target:", blank_target.sum())
print("duplicate rows with real country target:", (~blank_target).sum())

# look at the ones that ARE real countries
print("\nsample of real-country duplicates:")
dups[~blank_target].sort_values(
    ["sanctioning_state_iso3","sanctioned_state_iso3","year"]
)[["sanctioning_state_iso3","sanctioned_state_iso3","year",
   *["arms","military","trade","financial","travel","other"]]].head(20)

duplicate rows with blank target: 7141
duplicate rows with real country target: 0

sample of real-country duplicates:


,sanctioning_state_iso3,sanctioned_state_iso3,year,arms,military,trade,financial,travel,other


So, duplicates are only a problem for terrorist organizations. Let's drop all terrorist organizations - they are not countries.

In [ ]:
# drop non-state (terrorist-org) targets: no ISO3, outside a bilateral-trade gravity model
before = data_sanc.shape[0]
data_sanc = data_sanc[data_sanc["sanctioned_state_iso3"] != ""].copy()
print(f"dropped {before - data_sanc.shape[0]} non-state-target rows")

# confirm the duplicate problem is gone on real country-pairs
print("dup keys now:",
      data_sanc.duplicated(
          ["sanctioning_state_iso3","sanctioned_state_iso3","year"]).sum())   # expect 0

dropped 8865 non-state-target rows
dup keys now: 0


## Gravity dataset

Now to data_grav pre-save checks.

In [ ]:
# 1. Duplicate dyad-years (already saw 0, but confirm on current frame)
print("grav dup keys:",
      data_grav.duplicated(["exp_iso3","imp_iso3","year"]).sum())

# 2. The outcome: how much of 'trade' is missing vs zero vs positive?
print("\ntrade NaN:", data_grav["trade"].isna().sum())
print("trade == 0:", (data_grav["trade"] == 0).sum())
print("trade < 0 :", (data_grav["trade"] < 0).sum())   # should be 0
print("trade > 0 :", (data_grav["trade"] > 0).sum())

# 3. Missing merge keys (a NaN iso/year can't join)
print("\nkey NaN:",
      data_grav[["exp_iso3","imp_iso3","year"]].isna().sum().to_dict())

# 4. Self-pairs (country with itself — drop if any)
print("self-pairs:", (data_grav["exp_iso3"]==data_grav["imp_iso3"]).sum())

# 5. Year range sanity
print("years:", data_grav["year"].min(), "-", data_grav["year"].max())

grav dup keys: 120285

trade NaN: 1019780
trade == 0: 0
trade < 0 : 0
trade > 0 : 694828

key NaN: {'exp_iso3': 0, 'imp_iso3': 0, 'year': 0}
self-pairs: 7290
years: 1995 - 2021


Okay. Duplicates and self-pairs are a problem.

#### Duplicates

In [ ]:
# are the "duplicate" rows identical, or do they differ in the id columns / trade?
dups_g = data_grav[data_grav.duplicated(["exp_iso3","imp_iso3","year"], keep=False)]
print("dup rows:", dups_g.shape[0])

# pick one duplicated triple and show ALL its columns
ex = dups_g.sort_values(["exp_iso3","imp_iso3","year"]).iloc[:6]
print(ex[["exp_iso3","imp_iso3","year","exp_id","imp_id","trade"]].to_string())

# how many dup rows are fully identical vs differ somewhere?
print("\nfully identical dup rows:",
      data_grav.duplicated().sum())

dup rows: 236196
    exp_iso3 imp_iso3  year exp_id imp_id  trade
162      ABW      ANT  1995    ABW  ANT.1    NaN
189      ABW      ANT  1995    ABW  ANT.2    NaN
163      ABW      ANT  1996    ABW  ANT.1    NaN
190      ABW      ANT  1996    ABW  ANT.2    NaN
164      ABW      ANT  1997    ABW  ANT.1    NaN
191      ABW      ANT  1997    ABW  ANT.2    NaN

fully identical dup rows: 0


CEPII splits some ISO codes into sub-entities (e.g. ANT.1 / ANT.2), creating apparent duplicate ISO-triples.

In [ ]:
#  Collapse to one row per
# (exp_iso3, imp_iso3, year), keeping a non-null trade value if any exists.
before = data_grav.shape[0]

data_grav = (data_grav
    .sort_values("trade", na_position="last")          # non-null trade sorts first
    .drop_duplicates(["exp_iso3","imp_iso3","year"], keep="first")
    .reset_index(drop=True))

print(f"{before} -> {data_grav.shape[0]} rows")
print("dup keys now:",
      data_grav.duplicated(["exp_iso3","imp_iso3","year"]).sum())   # expect 0

1714608 -> 1594323 rows
dup keys now: 0


#### Self-pairs
They are not needed for the project analysis. Let's drop them.

In [ ]:
data_grav = data_grav[data_grav["exp_iso3"] != data_grav["imp_iso3"]].reset_index(drop=True)
print("self-pairs now:", (data_grav["exp_iso3"]==data_grav["imp_iso3"]).sum())  # expect 0
print("final grav shape:", data_grav.shape)

self-pairs now: 0
final grav shape: (1587762, 29)


# Merge

In [ ]:
# ============================================================
# GSDB × Gravity merge — two directional specifications
#   MAIN : sender -> exporter, target -> importer
#          => effect of sanctions on the TARGET'S IMPORTS FROM THE SENDER
#   ROBUST: sender -> importer, target -> exporter  (reversed)
#          => effect of sanctions on the TARGET'S EXPORTS TO THE SENDER
# Everything except the join mapping is identical across the two.
# ============================================================

sanction_types = ["arms","military","trade","financial","travel","other"]

# ---- Shared GSDB prep (done ONCE, before either merge) ----
sanc_base = data_sanc.rename(columns={t: f"sanc_{t}" for t in sanction_types}).copy()
# keep only what we need for the merge + features (drop 'success' = post-hoc outcome, avoids leakage)
sanc_keep = ["case_id","sanctioning_state_iso3","sanctioned_state_iso3","year",
             *[f"sanc_{t}" for t in sanction_types],
             "descr_trade","target_mult","sender_mult","objective"]
sanc_base = sanc_base[sanc_keep]

# ---- one function, two mappings: guarantees identical processing ----
def build_merged(gravity, sanc_base, sender_role):
    """sender_role='exporter' -> MAIN spec; 'importer' -> reversed robustness spec."""
    s = sanc_base.copy()
    if sender_role == "exporter":
        s = s.rename(columns={"sanctioning_state_iso3":"exp_iso3",
                              "sanctioned_state_iso3":"imp_iso3"})
    elif sender_role == "importer":
        s = s.rename(columns={"sanctioning_state_iso3":"imp_iso3",
                              "sanctioned_state_iso3":"exp_iso3"})

    merged = gravity.merge(s, on=["exp_iso3","imp_iso3","year"], how="left")

    # sanction flags: NaN after left-join = no sanction that dyad-year -> 0
    flag_cols = [f"sanc_{t}" for t in sanction_types] + ["target_mult","sender_mult"]
    merged[flag_cols] = merged[flag_cols].fillna(0)
    merged["sanctioned_any"] = (merged[[f"sanc_{t}" for t in sanction_types]].sum(axis=1) > 0).astype(int)
    return merged

merged_main   = build_merged(data_grav, sanc_base, sender_role="exporter")   # primary
merged_robust = build_merged(data_grav, sanc_base, sender_role="importer")   # robustness

print("main:  ", merged_main.shape,   "sanctioned rows:", int(merged_main["sanctioned_any"].sum()))
print("robust:", merged_robust.shape, "sanctioned rows:", int(merged_robust["sanctioned_any"].sum()))

main:   (1587762, 41) sanctioned rows: 98535
robust: (1587762, 41) sanctioned rows: 98535


In [ ]:
# check dtypes match (both should be 'object' = string)
print(data_grav["exp_iso3"].dtype, data_sanc["sanctioning_state_iso3"].dtype)

# GSDB sender codes with no match in Gravity -> these rows get dropped
print(set(data_sanc["sanctioning_state_iso3"].unique()) - set(data_grav["exp_iso3"].unique()))

# same check, target side
print(set(data_sanc["sanctioned_state_iso3"].unique()) - set(data_grav["imp_iso3"].unique()))

object object
{'KSV', 'HVO', 'DHY', 'SVU'}
{'KSV', 'RHO', 'SVU'}


These are defunct countries:

 Sender side: SVU (USSR), DHY (Dahomey/Benin), HVO (Upper Volta/Burkina Faso), KSV (Kosovo).

 Target side: SVU, RHO (Rhodesia/Zimbabwe), KSV.

### Export

In [ ]:
%%time
merged_main.to_csv(export_path   + "merged_main_raw.csv",   index=False)
merged_robust.to_csv(export_path + "merged_robust_raw.csv", index=False)

CPU times: user 1min 39s, sys: 824 ms, total: 1min 40s
Wall time: 1min 49s
